In [1]:
import torch


In [2]:
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

class FirstTerm(nn.Module):
    def __init__(self, num_cell_types, num_of_clusters , embedding_dim=8):
        super().__init__()
        self.cell_embedding = nn.Embedding(num_cell_types, embedding_dim)
        # Initialize 10.9M weights (The Dials)
        feature_dim = 18 + embedding_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(feature_dim*2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        nn.init.normal_(self.edge_mlp[2].weight, mean=0.0, std=0.001)
        nn.init.constant_(self.edge_mlp[2].bias, 0.01)


        # 3. THE CLUSTER HEAD (The C-Matrix Generator)
        # This replaces the N x K parameter bottleneck.
        self.cluster_head = nn.Sequential(
            nn.Linear(18 + embedding_dim, 64), 
            nn.ReLU(),
            nn.Linear(64, num_of_clusters)      # Output is K clusters
        )



    def forward(self, X, X_cell_ids, num_nodes, p_indices, A_skip_csr, current_k, tau=1.0):
        
        X_cell_ids = X_cell_ids.squeeze()
        cell_features = self.cell_embedding(X_cell_ids)  # Shape: [num_nodes, embedding_dim]
        X_combined = torch.cat([X, cell_features], dim=1)  # Shape: [num_nodes, 18 + embedding_dim]

        src_features = X_combined[p_indices[0]]  # Shape: [num_edges, feature_dim]
        dst_features = X_combined[p_indices[1]]  # Shape: [num_edges, feature_dim]
        edge_inputs = torch.cat([src_features, dst_features], dim=1)  # Shape: [num_edges, feature_dim*2]

        dynamic_p_weights = self.edge_mlp(edge_inputs).squeeze(-1)
        safe_weights = F.softplus(dynamic_p_weights) 


        # Enforce P >= 0 and build sparse matrix
        P = torch.sparse_coo_tensor(p_indices, safe_weights, 
                                    (num_nodes, num_nodes)).coalesce()
        
        # Reconstruction: XP
        X_hat = torch.sparse.mm(P, X)
        
        # Loss: ||X - XP||
        error = X - X_hat
        loss1 = torch.mean(error**2)   

        # Pass all node features through the head
        logits = self.cluster_head(X_combined) # Shape: [n, k]

        logits = logits[:, :current_k]
        
        #TERM2
        #C matrix with probability distribution across clusters for each node
        C = F.gumbel_softmax(logits, tau=tau, hard=False) # Shape: [n, k]
        # C = F.softmax(logits, dim=-1)  # Ensure positivity for SDDMM

        p_vals = P.values()
        
        # 1. Sum across rows (dim=1) to get the total weight leaving each node
        row_sums = torch.sparse.sum(P, dim=1).to_dense()
        
        # 2. Expand row_sums to match the non-zero values 
        # P.indices()[0] contains the row index for every specific edge
        p_vals_norm = p_vals / (row_sums[P.indices()[0]] + 1e-8)
        
        # Rebuild using the exact same sorted indices
        P_norm = torch.sparse_coo_tensor(P.indices(), p_vals_norm, 
                                         (num_nodes, num_nodes)).coalesce()
        
        # 2. Convert to CSR format (Required for the CUDA SDDMM engine)
        P_csr = P_norm.to_sparse_csr()
        
        # 3. SDDMM Magic! 
        # beta=1.0, alpha=-1.0 calculates exactly: (1.0 * P_csr) - (1.0 * C @ C^T)
        # It ONLY calculates this at the 10.9M non-zero locations!
        diff_csr = torch.sparse.sampled_addmm(P_csr, C, C.t(), beta=1.0, alpha=-1.0)
        
        # 4. Square the differences and sum them
        loss2 = torch.sum(diff_csr.values() ** 2)


        #TERM3
        M = torch.matmul(C.t(), torch.sparse.mm(A_skip_csr, C))  # [k, n] @ [n, n] @ [n, k] -> [k, k]
        # 2. Normalize M into a probability distribution (M_tilde)
        M = torch.clamp(M, min=0)
        M_tilde = M / (M.sum() + 1e-8)
        loss3 = -torch.sum(M_tilde * torch.log(M_tilde + 1e-8))

        # 3. Calculate Shannon Entropy: -sum(p * log(p))
        # We only calculate for non-zero entries to avoid log(0)
        # loss3 = torch.sum(M_tilde * torch.log(M_tilde + 1e-8))

        

        alpha_1 = 1.0 
        alpha_2 = 1.0    
        alpha_3 = 1.0    
        loss = alpha_1* loss1 + (alpha_2 * loss2) +(alpha_3 * loss3) 


        return  loss, loss1 , loss2, loss3,  C , X_combined


import math
def get_tau(epoch):
    tau_start = 2.0
    tau_mid = 1.35   # Targets ~25% Confidence for the middle flatline
    tau_end = 0.85   # Targets ~40% Confidence for the final floor

    if epoch < 75:
        # Phase 1: Smooth descent to the middle flatline
        progress = epoch / 75.0
        return tau_mid + 0.5 * (tau_start - tau_mid) * (1 + math.cos(math.pi * progress))
        
    elif epoch < 150:
        # Phase 2: THE MIDDLE FLATLINE (Epoch 75 to 150)
        return tau_mid
        
    elif epoch < 200:
        # Phase 3: Smooth descent to the final floor
        progress = (epoch - 150) / 50.0
        return tau_end + 0.5 * (tau_mid - tau_end) * (1 + math.cos(math.pi * progress))
        
    else:
        # Phase 4: THE FINAL FLATLINE (Epoch 200 to 250+)
        return tau_end

In [3]:
#Graph Neural Network Inference on the compressed graph (X_tilde, A_tilde_skip) 

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GraphConv, to_hetero , global_mean_pool



# ==========================================
# 1. The Base Homogeneous SAGE Model
# ==========================================
class BaseSAGE(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        # Using (-1, -1) leverages PyG's lazy initialization 
        self.conv1 = GraphConv((-1, -1), hidden_dim)
        self.conv2 = GraphConv((-1, -1), hidden_dim)

    def forward(self, x, edge_index, edge_weight=None):
        x = F.elu(self.conv1(x, edge_index, edge_weight))
        x = F.elu(self.conv2(x, edge_index, edge_weight))
        return x

class HeteroCTS_GNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_params):
        super().__init__()
        
        # 1. Define the Heterogeneous Schema explicitly
        self.metadata = (
            ['supernode'], 
            [
                ('supernode', 'physical', 'supernode'), 
                ('supernode', 'timing', 'supernode')
            ]
        )
        
        # 2. Initial projection to get raw features into hidden_dim space
        self.proj = nn.Linear(input_dim, hidden_dim)
        
        # 3. Create the Heterogeneous GNN
        # 'sum' aggregation combines the messages from physical and timing paths at each node
        self.gnn = to_hetero(BaseSAGE(hidden_dim), self.metadata, aggr='sum')
        
        # 4. Task Heads 
        self.power_head = nn.Sequential(
            nn.Linear(hidden_dim + num_params, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # Wirelength: Expects mean + variance concatenated (size: 2 * hidden_dim)
        self.wl_head = nn.Sequential(
            nn.Linear(hidden_dim + num_params, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )  
        
        # Skew: Placeholder for Attention Pooling (Currently using Mean)
        self.skew_attn = nn.Linear(hidden_dim, 1) # Attention weights
        self.skew_head = nn.Sequential(
            nn.Linear(hidden_dim + num_params, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )


    def forward(self, x_dict, edge_index_dict,cts_params, edge_weight_dict=None):
        # Initial projection and GNN message passing
        x = {key: self.proj(feat) for key, feat in x_dict.items()}
        h_dict = self.gnn(x, edge_index_dict, edge_weight_dict)
        h = h_dict['supernode'] # [K, hidden_dim]

        # 2. Power: Mean Pool and Concatenate Knobs
        h_power = torch.mean(h, dim=0, keepdim=True) # [1, hidden_dim]
        h_power_combined = torch.cat([h_power, cts_params], dim=-1)
        power_pred = self.power_head(h_power_combined)

        # 3. Wirelength: Mean + Var Pool and Concatenate Knobs [cite: 198, 199]
        mu = h.mean(dim=0, keepdim=True)
        h_wl_combined = torch.cat([mu,  cts_params], dim=-1)
        wl_pred = self.wl_head(h_wl_combined)

        # 4. Skew: Attention Pool and Concatenate Knobs [cite: 190, 191]
        attn_scores = F.softmax(self.skew_attn(h), dim=0)
        h_skew = torch.sum(attn_scores * h, dim=0, keepdim=True)
        h_skew_combined = torch.cat([h_skew, cts_params], dim=-1)
        skew_pred = self.skew_head(h_skew_combined)

        return skew_pred, power_pred, wl_pred
    
 

/home/rain/CTS-Task-Aware-Clustering/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os , random
processed_dir = "processed_graphs"
# List all processed file names
design_files = [f for f in os.listdir(processed_dir) if f.endswith('.pt')]

grouped_designs = {}
for f in design_files:
    # Splits "aes_run_20260306..." to extract just "aes"
    base_name = f.split('_run_')[0] 
    if base_name not in grouped_designs:
        grouped_designs[base_name] = []
    grouped_designs[base_name].append(f)

# 2. Pick exactly 5 from each category
balanced_files = []
for name, files in grouped_designs.items():
    random.shuffle(files)          # Shuffle within the category
    selected = files[:5]           # Take exactly 5 (or less if a category has < 5)
    balanced_files.extend(selected)

# 3. Final shuffle so the epoch isn't ordered by chip type
random.shuffle(balanced_files)
design_files = balanced_files      # Overwrite the main list

print(f"🔥 Fast-Tracking on {len(design_files)} perfectly balanced designs!")
# Quick check to prove it's balanced
print("Balance Check:", {name: len([x for x in design_files if name in x]) for name in grouped_designs.keys()})


🔥 Fast-Tracking on 20 perfectly balanced designs!
Balance Check: {'sha256': 5, 'aes': 5, 'ethmac': 5, 'picorv32': 5}


In [5]:
import torch
from helper import get_compressed_graph, relative_masking

# Make sure FirstTerm is imported or defined in your script
# from your_model_file import FirstTerm 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 4. INITIALIZE AND FREEZE PHASE 1 
# ==========================================
print("\n❄️ Loading and Freezing Phase 1 Brain...")
# Initialize with the exact dimensions used in your extraction
phase1_model = FirstTerm(num_cell_types=425, num_of_clusters=1000).to(device)

# Load the Epoch 250 weights
phase1_model.load_state_dict(torch.load("phase1_clustering_ep250.pt", map_location=device))
phase1_model.eval() # CRITICAL: Disables dropout and locks batchnorm layers

# Freeze the gradients to save massive amounts of VRAM
for param in phase1_model.parameters():
    param.requires_grad = False

print("✅ Phase 1 locked and loaded.")

# ==========================================
# 5. TEST THE COMPRESSION PIPELINE
# ==========================================
print("\n🔍 Testing compression on the first balanced design...")

# Pick just the first design to verify the pipeline
test_design = design_files[0]
data = torch.load(os.path.join(processed_dir, test_design), map_location=device)

print(f"Loaded: {test_design} | Original Nodes: {data['num_nodes']}")

# We use torch.no_grad() because Phase 1 is strictly a feature-extractor now
with torch.no_grad():
    # 1. Run the frozen Phase 1 model to get the clustering matrix C
    # We pass tau=0.85 as that was your locked temperature at Epoch 250
    loss, l1, l2, l3, C, X_combined = phase1_model(
        data['X'], 
        data['X_cell_ids'].squeeze(), 
        data['num_nodes'], 
        data['p_indices'], 
        data['A_skip_csr'], 
        data['current_k'], 
        tau=0.85 
    )

    # 2. Compress the Raw Graph into the Super-Graph
    # Notice we are passing data['raw_areas'] to maintain pure physical mass!
    X_tilde, A_tilde_skip, A_wire = get_compressed_graph(
        X_combined, 
        C, 
        data['A_skip_csr'], 
        data['A_wire_csr'], 
        data['raw_areas']
    )
    
    # 3. Clean up the fractional edge noise using your masking function
    wire_edge_index, wire_edge_weight = relative_masking(A_wire, threshold=0.10)
    skip_edge_index, skip_edge_weight = relative_masking(A_tilde_skip, threshold=0.10)

# Print the final shapes to guarantee the hand-off to the GNN is perfect
# ==========================================
# 6. SUPERNODE FEATURE INSPECTION (X_tilde)
# ==========================================
print("\n" + "="*50)
print("🔬 INSPECTING X_TILDE (SUPERNODE FEATURES)")
print("="*50)

# Extract the specific physical columns
x_centroids = X_tilde[:, 0]
y_centroids = X_tilde[:, 1]
counts = X_tilde[:, -2]  # The raw_areas sum
spreads = X_tilde[:, -1] # The spatial variance

print(f"📍 Centroids (X, Y) [Should be roughly between -2.0 and +2.0]:")
print(f"   -> X: Min {x_centroids.min().item():>7.3f} | Max {x_centroids.max().item():>7.3f} | Mean {x_centroids.mean().item():>7.3f}")
print(f"   -> Y: Min {y_centroids.min().item():>7.3f} | Max {y_centroids.max().item():>7.3f} | Mean {y_centroids.mean().item():>7.3f}")

print(f"\n🧱 Cluster Mass / 'Count' (Total True Physical Area):")
print(f"   -> Min:  {counts.min().item():.2f}")
print(f"   -> Max:  {counts.max().item():.2f}")
print(f"   -> Mean: {counts.mean().item():.2f}")

print(f"\n🌌 Spatial Spread (Bounding Box Size):")
print(f"   -> Min:  {spreads.min().item():.4f}")
print(f"   -> Max:  {spreads.max().item():.4f}")
print(f"   -> Mean: {spreads.mean().item():.4f}")

# Optional: Look at a random normal feature, like Average Pin Cap (assume it's column 3 or 4)
# We will just grab column 3 as an example of a standardized feature
random_feature = X_tilde[:, 3]
print(f"\n⚡ Standardized Feature Example (Column 3):")
print(f"   -> Min {random_feature.min().item():.3f} | Max {random_feature.max().item():.3f} | Mean {random_feature.mean().item():.3f}")
print("="*50)


❄️ Loading and Freezing Phase 1 Brain...
✅ Phase 1 locked and loaded.

🔍 Testing compression on the first balanced design...


/home/rain/CTS-Task-Aware-Clustering/venv/lib/python3.12/site-packages/torch/_utils.py:361: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  result = torch.sparse_compressed_tensor(


Loaded: aes_run_20260306_180433.pt | Original Nodes: 20918

🔬 INSPECTING X_TILDE (SUPERNODE FEATURES)
📍 Centroids (X, Y) [Should be roughly between -2.0 and +2.0]:
   -> X: Min  -0.283 | Max   0.314 | Mean  -0.000
   -> Y: Min  -0.318 | Max   0.308 | Mean   0.000

🧱 Cluster Mass / 'Count' (Total True Physical Area):
   -> Min:  154.11
   -> Max:  372.07
   -> Mean: 232.50

🌌 Spatial Spread (Bounding Box Size):
   -> Min:  1.2881
   -> Max:  1.5539
   -> Mean: 1.4065

⚡ Standardized Feature Example (Column 3):
   -> Min -0.314 | Max 0.283 | Mean 0.000


In [6]:
# ==========================================
# FULL FEATURE X-RAY FOR SUPERNODE 278
# ==========================================
target_cluster = 278
print(f"\n🔍 FULL FEATURE X-RAY: SUPERNODE {target_cluster}")
print("="*50)

# Get the features for just this supernode
sn_features = X_tilde[target_cluster].cpu().numpy()

# Note: Adjust these names based on your actual extraction order!
feature_names = [
    "0: X Coordinate (Avg)", "1: Y Coordinate (Avg)", 
    "2: Base Feature 3", "3: Base Feature 4", "4: Base Feature 5", 
    "5: Base Feature 6", "6: Cell Area (Sum)", "7: Base Feature 8",
    "8: Base Feature 9", "9: Base Feature 10", "10: Is_FF (Boolean)", 
    # ... (Fill in the rest up to 18 based on your raw X matrix)
]

print("--- Base Features (from C_norm.t() @ X) ---")
for i in range(18): # Assuming first 18 are from X_tilde_base
    name = feature_names[i] if i < len(feature_names) else f"{i}: Unknown Feature"
    print(f"  Feature {name:<25}: {sn_features[i]:>8.4f}")

print("\n--- Phase 2 Added Features ---")
# If you appended Count and Spread at the end:
print(f"  Feature 18: Total Area/Count     : {sn_features[-2]:>8.4f}")
print(f"  Feature 19: Physical Spread (Var): {sn_features[-1]:>8.4f}")


🔍 FULL FEATURE X-RAY: SUPERNODE 278
--- Base Features (from C_norm.t() @ X) ---
  Feature 0: X Coordinate (Avg)    :  -0.0758
  Feature 1: Y Coordinate (Avg)    :   0.1053
  Feature 2: Base Feature 3        :  -0.0758
  Feature 3: Base Feature 4        :   0.0758
  Feature 4: Base Feature 5        :  -0.1053
  Feature 5: Base Feature 6        :   0.1053
  Feature 6: Cell Area (Sum)       :  -0.1239
  Feature 7: Base Feature 8        :  -0.1112
  Feature 8: Base Feature 9        :  -0.2293
  Feature 9: Base Feature 10       :   0.1542
  Feature 10: Is_FF (Boolean)      :   0.1309
  Feature 11: Unknown Feature      :   0.3637
  Feature 12: Unknown Feature      :  -0.1660
  Feature 13: Unknown Feature      :  -0.1781
  Feature 14: Unknown Feature      :   0.0043
  Feature 15: Unknown Feature      :  -0.2300
  Feature 16: Unknown Feature      :  -0.2833
  Feature 17: Unknown Feature      :   0.2673

--- Phase 2 Added Features ---
  Feature 18: Total Area/Count     : 169.2453
  Feature 19: